**Cell #01**

# RAG11 Nutrition — Stage 1.2: Load Chunks to Supabase

Reads every `stage1_eda_output/sources/source_row-N.json` plus every `parent_chunk-N.json` /
`child_chunk-parentN-chunkM.json` written by `stage1_1_eda_extract_and_chunk.ipynb`, embeds each child chunk with
Voyage AI, and upserts rows into `rag11_data_sources`, `rag11_chunks_parent_table` and `rag11_chunks_child_table`
(in that order: parent/child rows carry a foreign key onto `rag11_data_sources`).

**All the code lives in `reusable_code/stage1/load.py`** (shared loaders and row builders in `stage1/common.py`); this
notebook calls its steps one by one. The whole pipeline without notebooks: `./run_stage1_all.command`.

**Before running**: `sql/create_sql_tables.sql` must have been run once in the Supabase SQL Editor.

Row shape (identical to the schema): `rowGUID` / `rowOwnerGUID` / `rowParentGUID` / `orderInList` / `rowJSON`:

- `rowGUID` — deterministic `uuid5` of a stable business key (a source's Drive `file_id`, or a chunk's own
  `parent_id`/`child_id`), so re-running **upserts** instead of duplicating.
- `rowOwnerGUID` — the owning source's `rag11_data_sources` row (its own `rowGUID` for a source row).
- `rowParentGUID` — `None` for sources and parent chunks; for a child, the uuid5 of its parent's `parent_id`, so it
  always matches the parent row and satisfies the foreign key.
- `orderInList` — the number parsed out of the chunk's **file name** (not trusted from the JSON).
- `rowJSON` — the file's JSON content, verbatim.

In [ ]:
# Cell #02
%pip install -q -r ../requirements.txt

**Cell #03**

## Clients

Supabase and Voyage clients from `.env` (this stage needs no Anthropic key). It also reports the process's open-file
limit: if it is low, repeated `[Errno 35] Resource temporarily unavailable` during upserts is almost certainly why
(`./run_stage1_all.command` raises it for you; in Jupyter/PyCharm run `stage1_0_run_mac_settings.command` once and
fully restart the app).

In [ ]:
# Cell #04
from reusable_code.stage1 import load as s12
from reusable_code.stage1.common import load_local_data

ctx = s12.LoaderContext.create()

**Cell #05**

## Load the local chunk files

Sources are the ones that have a manifest row; a stale `sourceN` folder left over from an older run is ignored (and
reported), never loaded as if it were current.

In [ ]:
# Cell #06
local = load_local_data(ctx.paths)

**Cell #07**

## Upserts are checkpointed and content-aware

Every successfully upserted row is recorded in `stage1_eda_output/_checkpoints/<table>_upserted_row_guids.json` as
`rowGUID -> fingerprint of the row's content` (the embedding is not part of the fingerprint: it is a pure function of
the chunk text). A re-run skips a row only if its fingerprint is unchanged, so:

- if a big load dies partway through, re-running only pushes what didn't make it;
- if you re-ran stage 1.1 and some chunks changed, only the changed rows are re-embedded and re-sent.

An older checkpoint (a plain list of ids, no fingerprints) means "contents unknown": every row is re-upserted once.
**If you truncate the Supabase tables** (`sql/delete_chunks_data.sql`), also delete `stage1_eda_output/_checkpoints/`.

**Cell #08**

## Sources — `rag11_data_sources`

Upserted *before* parents, since parent and child rows carry a foreign key onto `rag11_data_sources.rowGUID`.

In [ ]:
# Cell #09
s12.upsert_sources(ctx, local)

**Cell #10**

## Parents

Upserted *before* children, since the child table's `rowParentGUID` is a foreign key onto the parent table.

In [ ]:
# Cell #11
s12.upsert_parents(ctx, local)

**Cell #12**

## Children — embed only what is new or changed, then upsert

Voyage `voyage-3` embeddings (`input_type="document"`, 1024 dimensions), in parallel batches of 64 with retry; all
pending chunks across all sources go in one combined sweep so every batch is full-sized.

In [ ]:
# Cell #13
s12.upsert_children(ctx, local)

**Cell #14**

## Orphans in Supabase (rows with no local file)

An upsert never deletes: if stage 1.1 now produces fewer sections/chunks than before, the old rows stay in Supabase
and `stage1_9_eda_verify_all_data.ipynb` reports them as orphans. This cell lists them; set `PRUNE_ORPHANS = True` and
re-run it to delete them. Safety rails: only parent/child rows of a source present in the local output are
considered, source rows are never deleted here, and nothing is deleted while `PRUNE_ORPHANS` is `False`.

In [ ]:
# Cell #15
PRUNE_ORPHANS = False   # True = actually delete the orphaned rows listed

s12.prune_orphans(ctx, local, prune=PRUNE_ORPHANS)
print("Stage 1.2 ingestion complete.")